# Mount *Drive* italicized text

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Install and Import

In [ ]:
import pandas as pd
import numpy as np

# Paths — adjust if your CSV is in a different folder
RAW_CSV     = '/content/drive/MyDrive/DMIF/master_dataset.csv'
CLEAN_CSV   = '/content/drive/MyDrive/DMIF/master_dataset_clean.csv'
LOCAL_RAW   = '/content/master_dataset_raw.csv'
LOCAL_CLEAN = '/content/master_dataset_clean.csv'

# Copy to Local and Load

In [ ]:
import shutil

print("Copying CSV to local disk...")
shutil.copy(RAW_CSV, LOCAL_RAW)

df = pd.read_csv(LOCAL_RAW)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Copying CSV to local disk...
Loaded: 1199424 rows, 84 columns
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Trade_Volume', 'Share_Volume', 'Turnover', 'MA_24', 'MA_30', 'MA_500', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Lower', 'BB_Middle', 'BB_Width', 'SAR', 'SAR_Trend', 'Fib_0236', 'Fib_0382', 'Fib_05', 'Fib_0618', 'Fib_0786', 'Volume_MA_10', 'Volume_Ratio', 'Log_Volume', 'Daily_Return', 'Log_Return', 'HL_Range', 'OC_Range', 'Volatility_10', 'W_High', 'W_Low', 'W_Close', 'W_Volume', 'W_Turnover', 'W_Days', 'W_RSI', 'W_MACD', 'W_MA_10', 'W_MA_20', 'W_Return', 'W_Volatility', 'M_High', 'M_Low', 'M_Close', 'M_Volume', 'M_Turnover', 'M_Days', 'M_RSI', 'M_MACD', 'M_MA_6', 'M_MA_12', 'M_Return', 'M_Volatility', 'Q_High', 'Q_Low', 'Q_Close', 'Q_Volume', 'Q_Turnover', 'Q_Days', 'Q_RSI', 'Q_MACD', 'Q_MA_4', 'Q_MA_8', 'Q_Return', 'Q_Volatility', 'Y_High', 'Y_Low', 'Y_Close', 'Y_Volume', 'Y_Turnover', 'Y_Days', 'Y_RSI', 'Y_MACD', 'Y_MA_3', 'Y_MA_5', 'Y_Return', 'Y_Volatil

# Diagnostic

In [ ]:
print("="*60)
print("DIAGNOSTIC REPORT")
print("="*60)

print(f"\nTotal rows    : {len(df)}")
print(f"Total columns : {len(df.columns)}")

print("\n--- Null counts per column ---")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

print("\n--- Companies in dataset ---")
print(f"Total companies: {df['Company_Code'].nunique()}")
print(sorted(df['Company_Code'].unique()))

print("\n--- Rows per company (sorted least to most) ---")
rows_per_company = df.groupby('Company_Code').size().sort_values()
print(rows_per_company)

print("\n--- Companies with LESS than 31 rows (unusable) ---")
print(rows_per_company[rows_per_company < 31])

print("\n--- Infinity values per column ---")
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
print(inf_counts[inf_counts > 0])

print("\n--- Sample rows with nulls ---")
print(df[df.isnull().any(axis=1)].head(5))

DIAGNOSTIC REPORT

Total rows    : 1199424
Total columns : 84

--- Null counts per column ---
BB_Width              4
Fib_0236           1169
Fib_0382           1169
Fib_05             1169
Fib_0618           1169
Fib_0786           1169
Volume_Ratio         50
Daily_Return       1047
Log_Return         1047
HL_Range           3422
OC_Range           3413
Volatility_10       992
W_High              421
W_Low              2067
W_Close             415
W_Volume            415
W_Turnover          415
W_Days              415
W_RSI               415
W_MACD              415
W_MA_10             415
W_MA_20             415
W_Return           2054
W_Volatility       2646
M_High             1994
M_Low              2623
M_Close            1993
M_Volume           1993
M_Turnover         1993
M_Days             1993
M_RSI              1993
M_MACD             1993
M_MA_6             1993
M_MA_12            1993
M_Return           6364
M_Volatility       9778
Q_High           125409
Q_Low            1

# Targeted Cleaning

In [ ]:
print("Starting targeted cleaning...")
original_rows = len(df)

# ── 1. Sort chronologically per company ───────────────────────
df = df.sort_values(['Company_Code', 'Date']).reset_index(drop=True)
print("✓ Sorted by company and date")

# ── 2. Replace infinity values with NaN ───────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print("✓ Infinity values replaced with NaN")

# ── 3. Forward fill then back fill per company ────────────────
# This fixes ALL the warmup NaNs in W_, M_, Q_, Y_ columns
# and the small nulls in BB_Width, Fib, Volume_Ratio etc.
print("Filling NaN values per company (this may take a minute)...")

fill_cols = [c for c in df.columns
             if c not in ['Date', 'Company_Code', 'Target']]

filled_groups = []
for company, group in df.groupby('Company_Code'):
    group = group.copy()
    group[fill_cols] = (
        group[fill_cols]
        .ffill()
        .bfill()
    )
    filled_groups.append(group)

df = pd.concat(filled_groups).reset_index(drop=True)
print("✓ NaN filling complete")

# ── 4. Clip extreme values in return/volatility columns ───────
# Clip to reasonable range — stock returns beyond ±500% are errors
return_cols = [c for c in df.columns if 'Return' in c or 'Volatility' in c]
df[return_cols] = df[return_cols].clip(-5.0, 5.0)
print(f"✓ Clipped {len(return_cols)} return/volatility columns to [-5, 5]")

# ── 5. Clip all other numeric columns to remove outliers ──────
other_numeric = [c for c in numeric_cols
                 if c not in return_cols
                 and c not in ['Open','High','Low','Close','Volume',
                               'Trade_Volume','Share_Volume','Turnover']]
df[other_numeric] = df[other_numeric].clip(-1e6, 1e6)
print("✓ Clipped other numeric columns")

# ── 6. Select first 80 companies alphabetically ───────────────
all_companies_sorted = sorted(df['Company_Code'].unique())
top_80 = all_companies_sorted[:80]
print(f"\nFirst 80 companies selected:")
print(top_80)

df_80 = df[df['Company_Code'].isin(top_80)].reset_index(drop=True)
print(f"\n✓ Filtered to 80 companies")
print(f"  Total rows in 80-company dataset: {len(df_80)}")

# ── 7. Final null check ───────────────────────────────────────
remaining = df_80.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) > 0:
    print(f"\n⚠ Remaining nulls (only Target last-row NaNs are acceptable):")
    print(remaining)
else:
    print("\n✓ Zero nulls remaining")

print(f"\nCleaning complete.")
print(f"Original rows  : {original_rows}")
print(f"80-company rows: {len(df_80)}")
print(f"Companies      : {df_80['Company_Code'].nunique()}")
print(f"\nFirst 80 companies:")
for i, c in enumerate(top_80):
    print(f"  {i+1:2}. {c}")

Starting targeted cleaning...
✓ Sorted by company and date
✓ Infinity values replaced with NaN
Filling NaN values per company (this may take a minute)...
✓ NaN filling complete
✓ Clipped 11 return/volatility columns to [-5, 5]
✓ Clipped other numeric columns

First 80 companies selected:
['AAF.N', 'AAIC.N', 'ABAN.N', 'ABL.N', 'ACAP.N', 'ACME.N', 'AFS.N', 'AGAL.N', 'AGPL.N', 'AGST.N', 'AGST.X', 'AHPL.N', 'AHUN.N', 'AINS.N', 'ALLI.N', 'ALUM.N', 'AMF.N', 'AMSL.N', 'APLA.N', 'ASCO.N', 'ASHO.N', 'ASIR.N', 'ASIY.N', 'ASPH.N', 'ATL.N', 'ATLL.N', 'AUTO.N', 'BALA.N', 'BBH.N', 'BERU.N', 'BFL.N', 'BFN.N', 'BIL.N', 'BOGA.N', 'BOPL.N', 'BPPL.N', 'BREW.N', 'BRR.N', 'BRWN.N', 'BUKI.N', 'CABO.N', 'CALF.N', 'CALH.N', 'CALT.N', 'CARE.N', 'CARG.N', 'CARS.N', 'CBNK.N', 'CCS.N', 'CDB.N', 'CDB.X', 'CERA.N', 'CFI.N', 'CFIN.N', 'CFLB.N', 'CFVF.N', 'CHL.N', 'CHL.X', 'CHMX.N', 'CHOT.N', 'CIC.N', 'CIC.X', 'CIND.N', 'CINS.N', 'CINS.X', 'CINV.N', 'CIT.N', 'CITH.N', 'CITW.N', 'CLND.N', 'COCO.N', 'COCO.X', 'COCR.N',

# Verify

In [ ]:
print("="*60)
print("POST-CLEANING VERIFICATION")
print("="*60)

print("\nNull counts (should be only Target):")
nulls = df_80.isnull().sum()
print(nulls[nulls > 0])

print("\nInfinity check:")
inf_check = np.isinf(
    df_80.select_dtypes(include=[np.number])
).sum()
print(inf_check[inf_check > 0] if inf_check.sum() > 0
      else "✓ No infinity values")

print("\nRows per company:")
print(df_80.groupby('Company_Code').size()
      .sort_values(ascending=False))

print("\nSample of cleaned data:")
print(df_80.head(3))

print("\nColumns available for heatmap:")
print(df_80.columns.tolist())

POST-CLEANING VERIFICATION

Null counts (should be only Target):
Y_High          687
Y_Low           687
Y_Close         687
Y_Volume        687
Y_Turnover      687
Y_Days          687
Y_RSI           687
Y_MACD          687
Y_MA_3          687
Y_MA_5          687
Y_Return        687
Y_Volatility    687
dtype: int64

Infinity check:
✓ No infinity values

Rows per company:
Company_Code
COMB.N    7837
CFIN.N    7111
ASIR.N    7018
ACAP.N    6981
CFVF.N    6868
          ... 
AGPL.N     594
AFS.N      588
CBNK.N     508
CALH.N     179
AGST.X     131
Length: 80, dtype: int64

Sample of cleaned data:
         Date  Open  High  Low  Close  Trade_Volume  Share_Volume  \
0  2012-01-12   3.0   4.3  3.0    4.2          2428      38538200   
1  2012-01-13   4.5   6.3  4.5    5.8          3132      40938200   
2  2012-01-17   6.3   8.7  6.3    8.5          3625      45760300   

      Turnover     MA_24     MA_30  ...  Y_Turnover  Y_Days  Y_RSI  Y_MACD  \
0  144153480.0  4.200000  4.200000  ...   

# Save Both Files

In [ ]:
import os

LOCAL_CLEAN_80  = '/content/master_dataset_clean_80.csv'
LOCAL_CLEAN_ALL = '/content/master_dataset_clean_all.csv'
DRIVE_CLEAN_80  = '/content/drive/MyDrive/DMIF/master_dataset_clean_80.csv'
DRIVE_CLEAN_ALL = '/content/drive/MyDrive/DMIF/master_dataset_clean_all.csv'

# Save 80-company version
print("Saving 80-company clean CSV...")
df_80.to_csv(LOCAL_CLEAN_80, index=False)
shutil.copy(LOCAL_CLEAN_80, DRIVE_CLEAN_80)
print(f"✓ 80-company CSV saved: {os.path.getsize(DRIVE_CLEAN_80)/1e6:.1f} MB")

# Save full clean version as backup
print("Saving full clean CSV (all 291 companies)...")
df.to_csv(LOCAL_CLEAN_ALL, index=False)
shutil.copy(LOCAL_CLEAN_ALL, DRIVE_CLEAN_ALL)
print(f"✓ Full clean CSV saved: {os.path.getsize(DRIVE_CLEAN_ALL)/1e6:.1f} MB")

print("\nDone. Use master_dataset_clean_80.csv for chart generation.")

Saving 80-company clean CSV...
✓ 80-company CSV saved: 319.0 MB
Saving full clean CSV (all 291 companies)...
✓ Full clean CSV saved: 1178.5 MB

Done. Use master_dataset_clean_80.csv for chart generation.
